# Results for model: meta-llama/Llama-3.3-70B-Instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
import numpy as np

# Load data
df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

# Convert to pandas dataframe for easier manipulation
df = df.to_pandas()

# Feature engineering
df['TOP_QUANTILE'] = df.groupby('date_id')['feature_00'].transform(lambda x: np.where(x > x.quantile(0.85), 1, 0))

# Define features and target
X = df.drop(['responder_6', 'date_id'], axis=1)
y = df['responder_6']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost regressor
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', max_depth=5, learning_rate=0.1, n_estimators=1000)
xgb_model.fit(X_train, y_train)

# Make predictions
y_pred = xgb_model.predict(X_test)